In [11]:
import pandas as pd
import numpy as np

In [12]:
train = pd.read_csv("./used_car_train_20200313.csv",sep = r'\s+')
testA = pd.read_csv("./used_car_testA_20200313.csv", sep = r'\s+')
testB = pd.read_csv("./used_car_testB_20200421.csv", sep = r'\s+')
print("训练集大小", train.shape)
print("测试集大小", testA.shape)

训练集大小 (150000, 31)
测试集大小 (50000, 30)


In [13]:
### 查看原始数据
print("train:",train.head())
print("test:", testA.head())
print("test:", testB.head())
print("*" * 100)

### 缺失值
missing = train.isnull().sum()
print(missing)

train:    SaleID    name   regDate  model  brand  bodyType  fuelType gearbox power  \
0       0     736  20040402   30.0    6.0       1.0       0.0     0.0    60   
1       1    2262  20030301   40.0    1.0       2.0       0.0     0.0     0   
2       2   14874  20040403  115.0   15.0       1.0       0.0     0.0   163   
3       3   71865  19960908  109.0   10.0       0.0       0.0     1.0   193   
4       4  111080  20120103  110.0    5.0       1.0       0.0     0.0    68   

  kilometer  ...       v_5       v_6       v_7       v_8       v_9      v_10  \
0      12.5  ...  0.235676  0.101988  0.129549  0.022816  0.097462 -2.881803   
1      15.0  ...  0.264777  0.121004  0.135731  0.026597  0.020582 -4.900482   
2      12.5  ...  0.251410  0.114912  0.165147  0.062173  0.027075 -4.846749   
3      15.0  ...  0.274293  0.110300  0.121964  0.033395  0.000000 -4.509599   
4       5.0  ...  0.228036  0.073205  0.091880  0.078819  0.121534 -1.896240   

       v_11      v_12      v_13      

# 数据预处理

### 缺失值处理

In [14]:
### notRepairedDamage属性中的缺失值以"-"表示，替换为nan
print(train['notRepairedDamage'].unique())
train['notRepairedDamage'] = train['notRepairedDamage'].replace('-', np.nan)
testA['notRepairedDamage'] = testA['notRepairedDamage'].replace('-', np.nan)
testB['notRepairedDamage'] = testB['notRepairedDamage'].replace('-', np.nan)

['0.0' '-' '1.0' ... '1882' '7753' '2159']


In [15]:
### 提取目标列
y_train = train['price']
train.drop('price', axis=1, inplace=True)

### 统一处理，避免不一致
train['data_type'] = 1       # 训练集标记为1
testA['data_type'] = 2       # 测试集A标记为2
testB['data_type'] = 3       # 测试集B标记为3

all_data = pd.concat([train, testA, testB], sort=False).reset_index(drop=True)

### 特征分类
# 真正的数值连续特征
num_cols = ['power', 'kilometer', 'v_0', 'v_1', 'v_2', 'v_3', 'v_4', 'v_5',
            'v_6', 'v_7', 'v_8', 'v_9', 'v_10', 'v_11', 'v_12', 'v_13', 'v_14']

# 本质是类别的特征
cat_cols = ['bodyType', 'fuelType', 'gearbox', 'notRepairedDamage', 
            'seller', 'offerType', 'brand', 'model', 'regionCode', 'name']

# 日期列
date_cols = ['regDate', 'creatDate']

### 类型转换
# 数值特征：强制转浮点型
for col in num_cols:
    all_data[col] = pd.to_numeric(all_data[col], errors='coerce').astype(float)

# 分类特征：强制转数值型
for col in cat_cols:
    all_data[col] = pd.to_numeric(all_data[col], errors='coerce')


### 缺失值处理
for col in all_data.columns:
    if col == 'is_train':
        continue
    if all_data[col].isnull().any():
        if col in num_cols:
            # 如果数值型特征为空，新建_ismissing特征列（是否为空本身可能也是一种信息），若为空则填1，反之填0
            all_data[col + '_ismissing'] = all_data[col].isnull().astype(int)
            all_data[col] = all_data[col].fillna(all_data[col].median())
        elif col in cat_cols:
            # 类别特征如果为空则填入-999，将其与正常类别区分
            all_data[col] = all_data[col].fillna(-999)

### 完成类型变量的转换
for col in cat_cols:
    all_data[col] = all_data[col].astype(int)


### 日期处理

In [16]:
for col in ['regDate', 'creatDate']:
    all_data[col] = pd.to_datetime(all_data[col], format='%Y%m%d', errors='coerce')
# 构造汽车使用天数
all_data['car_age_days'] = (all_data['creatDate'] - all_data['regDate']).dt.days

In [17]:
train = all_data[all_data['data_type'] == 1].drop('data_type', axis=1).reset_index(drop=True)
testA = all_data[all_data['data_type'] == 2].drop('data_type', axis=1).reset_index(drop=True)
testB = all_data[all_data['data_type'] == 3].drop('data_type', axis=1).reset_index(drop=True)

# 加回价格列
train['price'] = y_train.values

In [18]:
print(train.head())

   SaleID    name    regDate  model  brand  bodyType  fuelType  gearbox  \
0       0     736 2004-04-02     30      6         1         0        0   
1       1    2262 2003-03-01     40      1         2         0        0   
2       2   14874 2004-04-03    115     15         1         0        0   
3       3   71865 1996-09-08    109     10         0         0        1   
4       4  111080 2012-01-03    110      5         1         0        0   

   power  kilometer  ...      v_12      v_13      v_14  power_ismissing  \
0   60.0       12.5  ... -2.420821  0.795292  0.914762                0   
1    0.0       15.0  ... -1.030483 -1.722674  0.245522                0   
2  163.0       12.5  ...  1.565330 -0.832687 -0.229963                0   
3  193.0       15.0  ... -0.501868 -2.438353 -0.478699                0   
4   68.0        5.0  ...  0.931110  2.834518  1.923482                0   

  kilometer_ismissing  v_12_ismissing  v_13_ismissing  v_14_ismissing  \
0                   0    

### 排除掉不符合题目要求的行 

In [19]:
# 定义题目要求的合法取值
LEGAL_RULES = {
    "bodyType": [0,1,2,3,4,5,6,7,-999],
    "fuelType": [0,1,2,3,4,5,6,-999],
    "gearbox": [0,1,-999],
    "notRepairedDamage": [0,1,-999],
    "seller": [0,1,-999],
    "offerType": [0,1,-999],
}
POWER_RANGE = (0, 600)

### 清洗训练集
train = train[
    train["bodyType"].isin(LEGAL_RULES["bodyType"]) &
    train["fuelType"].isin(LEGAL_RULES["fuelType"]) &
    train["gearbox"].isin(LEGAL_RULES["gearbox"]) &
    train["notRepairedDamage"].isin(LEGAL_RULES["notRepairedDamage"]) &
    train["seller"].isin(LEGAL_RULES["seller"]) &
    train["offerType"].isin(LEGAL_RULES["offerType"]) &
    train["power"].between(*POWER_RANGE)
].reset_index(drop=True)

### 清洗测试集
testA["power"] = testA["power"].clip(*POWER_RANGE)
testB["power"] = testB["power"].clip(*POWER_RANGE)

### 结果一览
print("筛选约束完成！")
print("清洗后训练集形状：", train.shape)
print("清洗后测试集A形状：", testA.shape)
print("清洗后测试集B形状：", testB.shape)

筛选约束完成！
清洗后训练集形状： (135770, 37)
清洗后测试集A形状： (50000, 36)
清洗后测试集B形状： (50000, 36)


In [20]:
### 保存数据
train.to_pickle("train_processed.pkl")   
testA.to_pickle("testA_processed.pkl")  
testB.to_pickle("testB_processed.pkl")  